# ⚽ Calculadora de escenarios — Mundial 2026 (API + análisis)

Trae los datos **solos** desde una API y te deja analizar a fondo cada grupo, con el
**desempate olímpico** de FIFA 2026 y los **mejores 8 terceros**.

**Qué podés hacer:**
- 🌐 Cargar todo automático desde la API (sin escribir nada).
- 📊 Tabla actual y panorama (clasificado / en disputa / eliminado).
- ❓ Qué necesita un equipo para 1º, clasificar directo, o quedar 3º.
- 🎯 **Elegir un puesto puntual y ver qué resultados se necesitan** (avisa si es imposible).
- 🔀 "¿Qué pasa si…?": fijás resultados y ves cómo queda el grupo.
- 🎲 Probabilidades estimadas (modelo de goles).
- 📈 Distribución de puestos y mejor/peor caso de cada equipo.
- 🏆 Torneo completo + tabla de mejores terceros.

**Cómo usar:** *Entorno de ejecución → Ejecutar todo*. Después usá los botones. La carga por
API está en la sección 1; si preferís, también podés pegar o cargar a mano (sección 9).

## ⚙️ Motor (ejecutar una vez)

Desempate Mundial 2026: **puntos** → entre los empatados **pts/dif. de gol/goles** del mano
a mano → **dif. de gol** y **goles** generales → fair play → ranking FIFA.

In [ ]:
from itertools import product, combinations
import pandas as pd, numpy as np, re

try:
    import ipywidgets as W
    from IPython.display import display, clear_output, HTML
    _HAY_WIDGETS = True
except Exception:
    _HAY_WIDGETS = False

def jugados_lista(dic): return [(l, v, gl, gv) for (l, v), (gl, gv) in dic.items()]
def fixture_completo(equipos): return list(combinations(equipos, 2))
def detectar_pendientes(equipos, dic):
    jug = {frozenset(p) for p in dic}
    return [p for p in fixture_completo(equipos) if frozenset(p) not in jug]

def _stats(equipos, partidos):
    st = {e: {"pts": 0, "gf": 0, "ga": 0, "pj": 0} for e in equipos}
    for l, v, gl, gv in partidos:
        st[l]["gf"] += gl; st[l]["ga"] += gv; st[l]["pj"] += 1
        st[v]["gf"] += gv; st[v]["ga"] += gl; st[v]["pj"] += 1
        if gl > gv: st[l]["pts"] += 3
        elif gl < gv: st[v]["pts"] += 3
        else: st[l]["pts"] += 1; st[v]["pts"] += 1
    for e in st: st[e]["dg"] = st[e]["gf"] - st[e]["ga"]
    return st

def _stats_entre(teams, partidos):
    ts = set(teams); st = {e: {"pts": 0, "gf": 0, "ga": 0} for e in teams}
    for l, v, gl, gv in partidos:
        if l in ts and v in ts:
            st[l]["gf"] += gl; st[l]["ga"] += gv; st[v]["gf"] += gv; st[v]["ga"] += gl
            if gl > gv: st[l]["pts"] += 3
            elif gl < gv: st[v]["pts"] += 3
            else: st[l]["pts"] += 1; st[v]["pts"] += 1
    for e in st: st[e]["dg"] = st[e]["gf"] - st[e]["ga"]
    return st

CRITERIOS = ["h2h_pts", "h2h_dg", "h2h_gf", "dg", "gf"]  # desempate por defecto (olímpico FIFA 2026)
PRESETS = {
    "Olímpico — mano a mano primero (FIFA, Euro, La Liga, Serie A)": ["h2h_pts","h2h_dg","h2h_gf","dg","gf"],
    "Diferencia de gol primero (Premier, Bundesliga, Champions fase liga)": ["dg","gf"],
    "Solo puntos (sin desempate fino)": [],
}
def usar_desempate(criterios):
    global CRITERIOS; CRITERIOS = list(criterios)

def _resolver(teams, partidos, overall, fair_play, ranking):
    if len(teams) <= 1: return list(teams)
    h = _stats_entre(teams, partidos) if any(c.startswith("h2h") for c in CRITERIOS) else None
    def val(c):
        if c == "h2h_pts": return {e: h[e]["pts"] for e in teams}
        if c == "h2h_dg":  return {e: h[e]["dg"] for e in teams}
        if c == "h2h_gf":  return {e: h[e]["gf"] for e in teams}
        if c == "dg":      return {e: overall[e]["dg"] for e in teams}
        if c == "gf":      return {e: overall[e]["gf"] for e in teams}
        if c == "fair_play" and fair_play is not None: return {e: fair_play.get(e, 0) for e in teams}
        if c == "ranking" and ranking is not None: return {e: -ranking.get(e, 9999) for e in teams}
        return None
    for c in CRITERIOS:
        vals = val(c)
        if vals is None: continue
        if len(set(vals.values())) > 1:
            out = []
            for v in sorted(set(vals.values()), reverse=True):
                out += _resolver([e for e in teams if vals[e] == v], partidos, overall, fair_play, ranking)
            return out
    return sorted(teams)

def _orden(equipos, partidos, fair_play=None, ranking=None):
    overall = _stats(equipos, partidos); porpts = {}
    for e in equipos: porpts.setdefault(overall[e]["pts"], []).append(e)
    orden = []
    for pts in sorted(porpts, reverse=True):
        orden += _resolver(porpts[pts], partidos, overall, fair_play, ranking)
    return orden, overall

def posiciones(equipos, partidos, fair_play=None, ranking=None):
    orden, _ = _orden(equipos, partidos, fair_play, ranking)
    return {e: i for i, e in enumerate(orden, 1)}

def tabla(equipos, partidos, fair_play=None, ranking=None):
    orden, ov = _orden(equipos, partidos, fair_play, ranking)
    return pd.DataFrame([{"Pos": i, "Equipo": e, "PJ": ov[e]["pj"], "PTS": ov[e]["pts"],
                          "GF": ov[e]["gf"], "GC": ov[e]["ga"], "DG": ov[e]["dg"]}
                         for i, e in enumerate(orden, 1)])

def simular(equipos, jugados, pendientes, resultados, fair_play=None, ranking=None):
    part = list(jugados) + [(l, v, gl, gv) for (l, v), (gl, gv) in zip(pendientes, resultados)]
    return tabla(equipos, part, fair_play, ranking)

def texto_resultados(pend, res):
    return " | ".join(f"{l} {gl}-{gv} {v}" for (l, v), (gl, gv) in zip(pend, res))

def elegir_max_goles(n_pend, tope=300000):
    for mg in (5, 4, 3, 2, 1):
        if (mg + 1) ** (2 * n_pend) <= tope: return mg
    return 1

def todos_los_escenarios(equipos, jugados, pendientes, max_goles=None, fair_play=None, ranking=None):
    if max_goles is None: max_goles = elegir_max_goles(len(pendientes))
    posib = list(product(range(max_goles + 1), repeat=2)); filas = []
    for res in product(posib, repeat=len(pendientes)):
        t = simular(equipos, jugados, pendientes, res, fair_play, ranking)
        fila = {"Resultados": texto_resultados(pendientes, res)}
        for i, ((l, v), (gl, gv)) in enumerate(zip(pendientes, res), 1):
            fila[f"P{i}_local"] = l; fila[f"P{i}_vis"] = v; fila[f"P{i}_gl"] = gl; fila[f"P{i}_gv"] = gv
        for _, r in t.iterrows():
            e = r["Equipo"]; fila[f"Pos {e}"] = r["Pos"]; fila[f"PTS {e}"] = r["PTS"]
            fila[f"DG {e}"] = r["DG"]; fila[f"GF {e}"] = r["GF"]
        filas.append(fila)
    return pd.DataFrame(filas)

print("Motor cargado ✓" + ("" if _HAY_WIDGETS else "  (sin ipywidgets)"))

In [ ]:
CAMPEON = "campeón"     # cómo llamar al 1º: "campeón" (liga) o, p.ej., "1º de la zona" (grupo)
DIRECTO = 2             # cuántos clasifican DIRECTO por grupo (top N)
MEJORES_TERCEROS = 8    # mejores terceros que clasifican en el torneo (0 = los terceros NO clasifican)

def _pd_de(equipo, pend): return [(i, l, v) for i, (l, v) in enumerate(pend, 1) if equipo in (l, v)]

def _res_propio(row, equipo, pend):
    et = []
    for i, l, v in _pd_de(equipo, pend):
        gl, gv = row[f"P{i}_gl"], row[f"P{i}_gv"]
        gf, gc = (gl, gv) if l == equipo else (gv, gl); riv = v if l == equipo else l
        et.append(f"le gana a {riv}" if gf > gc else (f"pierde con {riv}" if gf < gc else f"empata con {riv}"))
    return " y ".join(et)

def _res_otros(row, equipo, pend):
    et = []; mios = {i for i, _, _ in _pd_de(equipo, pend)}
    for i, (l, v) in enumerate(pend, 1):
        if i in mios: continue
        gl, gv = row[f"P{i}_gl"], row[f"P{i}_gv"]
        et.append(f"gana {l}" if gl > gv else (f"gana {v}" if gl < gv else f"empatan {l} y {v}"))
    return " y ".join(et) if et else "(no hay otros partidos)"

def _combo(row, pend):
    parts = []
    for i, (l, v) in enumerate(pend, 1):
        gl, gv = row[f"P{i}_gl"], row[f"P{i}_gv"]
        parts.append(f"gana {l}" if gl > gv else (f"gana {v}" if gl < gv else f"empatan {l} y {v}"))
    return " · ".join(parts)

def _margen_pend(eq, pend, row):
    m = 0; opp = None
    for i, l, v in _pd_de(eq, pend):
        gl, gv = row[f"P{i}_gl"], row[f"P{i}_gv"]
        m += (gl - gv) if l == eq else (gv - gl)
        opp = v if l == eq else l
    return m, opp

def _gol(k): return f"{abs(k)} gol" + ("es" if abs(k) != 1 else "")

def _detalle_gol(g2, equipo, pend):
    """Describe el caso 'depende del gol': contra quién y por cuánto margen."""
    fila = g2.iloc[0]; Pe = fila[f"PTS {equipo}"]
    teams = [c[4:] for c in g2.columns if c.startswith("PTS ")]
    rivales = [t for t in teams if t != equipo and g2[f"PTS {t}"].iloc[0] == Pe]
    if len(rivales) != 1:
        extra = f" (igualado en {int(Pe)} pts con {', '.join(rivales)})" if rivales else ""
        return f"depende de la diferencia de gol{extra}"
    riv = rivales[0]
    me0, opp = _margen_pend(equipo, pend, fila); mr0, _ = _margen_pend(riv, pend, fila)
    de = int(fila[f"DG {equipo}"]) - me0; dr = int(fila[f"DG {riv}"]) - mr0
    gap = dr - de; K = gap + 1; riv_pend = bool(_pd_de(riv, pend))
    solo_e = len(_pd_de(equipo, pend)) == 1; solo_r = len(_pd_de(riv, pend)) == 1
    if me0 > 0 and solo_e and solo_r:
        if K >= 2:
            return (f"necesita ganarle a {opp} por al menos {_gol(K)} más que {riv}; "
                    f"si gana por {_gol(K-1)} más, igualan en diferencia de gol y se define por los goles a favor")
        if K == 1:
            return (f"necesita ganarle a {opp} por al menos 1 gol más que {riv}; "
                    f"si ganan por la misma diferencia, igualan en DG y se define por los goles a favor")
        return (f"le alcanza con que su diferencia de gol final supere a la de {riv} (parte {_gol(-gap)} arriba); "
                f"si {riv} la empareja, se define por los goles a favor")
    if me0 > 0 and solo_e and not riv_pend and K >= 1:
        cola = (f"con {_gol(K-1)} igualan en DG y define los goles a favor" if K - 1 >= 1
                else "si igualan la DG, define los goles a favor")
        return f"necesita ganar por al menos {_gol(K)} para superar la diferencia de gol de {riv}; {cola}"
    return (f"necesita terminar con mejor diferencia de gol que {riv} "
            f"(hoy {equipo} {de:+d} y {riv} {dr:+d}); si igualan, se define por los goles a favor")

def situacion(equipo, esc, directo=None):
    directo = DIRECTO if directo is None else directo
    pos = esc[f"Pos {equipo}"]
    vivo = 3 if MEJORES_TERCEROS > 0 else directo   # hasta qué puesto seguís con chances
    return {"mejor": int(pos.min()), "peor": int(pos.max()), "total": len(esc),
            "n1": int((pos == 1).sum()), "ndir": int((pos <= directo).sum()),
            "ntercero": int((pos == 3).sum()), "ntop3": int((pos <= 3).sum()),
            "ya_1": bool((pos == 1).all()), "ya_directo": bool((pos <= directo).all()),
            "puede_1": bool((pos == 1).any()), "puede_directo": bool((pos <= directo).any()),
            "puede_tercero": bool((pos == 3).any()), "asegura_vivo": bool((pos <= vivo).all()),
            "eliminado": bool((pos > vivo).all()), "vivo": vivo, "directo": directo}

def que_necesita(equipo, esc, pend, objetivo="directo", directo=None, n=2):
    directo = DIRECTO if directo is None else directo
    pos = esc[f"Pos {equipo}"]
    T = sum(1 for c in esc.columns if c.startswith("Pos "))
    if objetivo in ("primero", "campeon"):
        ok = (pos == 1); meta = f"ser {CAMPEON} (salir 1º)"; verbo = f"es {CAMPEON}"
    elif objetivo == "top3":
        ok = (pos <= 3); meta = "quedar al menos 3º (chance de mejor tercero — depende de otros grupos)"; verbo = "queda 3º o mejor"
    elif objetivo == "tercero":
        ok = (pos == 3); meta = "quedar 3º (chance de mejor tercero — depende de otros grupos)"; verbo = "queda 3º"
    elif objetivo == "top":
        ok = (pos <= n); meta = f"clasificar (top {n})"; verbo = f"entra al top {n}"
    elif objetivo == "exacto":
        ok = (pos == n); meta = f"terminar exactamente {n}º"; verbo = f"queda {n}º"
    elif objetivo == "descenso":
        corte = T - n; ok = (pos <= corte); meta = f"salvarse del descenso (no quedar en los últimos {n})"; verbo = "se salva"
    else:
        ok = (pos <= directo); meta = f"clasificar directo (top {directo})"; verbo = "clasifica"
    df = esc.copy(); df["_p"] = df.apply(lambda r: _res_propio(r, equipo, pend), axis=1)
    df["_o"] = df.apply(lambda r: _res_otros(r, equipo, pend), axis=1); df["_ok"] = ok.values
    print(f"¿Qué necesita {equipo} para {meta}?\n")
    for prop, g in sorted(df.groupby("_p"), key=lambda kv: -kv[1]["_ok"].mean()):
        m, k = len(g), int(g["_ok"].sum())
        cab = "✅ SEGURO" if k == m else ("❌ IMPOSIBLE" if k == 0 else "⚠️  DEPENDE")
        print(f"• Si {equipo} {prop}:  {cab}")
        if 0 < k < m:
            for otros, g2 in sorted(g.groupby("_o"), key=lambda kv: -kv[1]["_ok"].mean()):
                n2, k2 = len(g2), int(g2["_ok"].sum())
                e = f"→ {verbo} ✅" if k2 == n2 else (f"→ no {verbo} ❌" if k2 == 0 else f"→ {_detalle_gol(g2, equipo, pend)} ⚠️")
                print(f"      · y {otros}:  {e}")
        print()

def apartado_terceros(equipo, esc, pend):
    """Apartado del 3º: solo aplica si el torneo admite mejores terceros."""
    if MEJORES_TERCEROS <= 0:
        return
    pos = esc[f"Pos {equipo}"]; n3 = int((pos == 3).sum())
    print("--- MEJOR TERCERO ---")
    if n3 == 0:
        print(f"{equipo} no termina 3º en ningún escenario.\n"); return
    print(f"⚠️  Quedar 3º NO asegura clasificar: entran los {MEJORES_TERCEROS} mejores terceros del")
    print("    torneo, así que depende de lo que pase en los OTROS grupos (se comparan por")
    print("    puntos, luego dif. de gol y goles).")
    print(f"{equipo} termina 3º en {n3}/{len(esc)} escenarios. Para saber si ese 3º entra, cargá")
    print("    los terceros de los demás grupos en la sección de 'mejores terceros'.\n")
    que_necesita(equipo, esc, pend, "tercero")

def analizar(equipo, esc, pend, directo=None):
    directo = DIRECTO if directo is None else directo
    s = situacion(equipo, esc, directo); hay3 = MEJORES_TERCEROS > 0
    print("=" * 56); print("  " + equipo.upper()); print("=" * 56)
    if s["ya_directo"]:
        print(f"🟢 YA CLASIFICÓ DIRECTO (siempre entre los {directo} primeros).")
    elif s["eliminado"]:
        print("🔴 ELIMINADO: no llega a zona de clasificación en ningún escenario.")
    else:
        print(f"Puede terminar entre {s['mejor']}º y {s['peor']}º.")
        if hay3 and s["asegura_vivo"]:
            print("🔵 Como mínimo termina 3º → siempre le queda la chance de mejor tercero (depende de otros grupos).")
        linea = f"   1º en {s['n1']}/{s['total']} · directo (top {directo}) en {s['ndir']}/{s['total']}"
        if hay3: linea += f" · 3º en {s['ntercero']}/{s['total']}"
        print(linea)
    print()
    if not s["eliminado"] and not s["ya_directo"]:
        que_necesita(equipo, esc, pend, "directo", directo)
    if s["puede_1"] and not s["ya_1"]:
        que_necesita(equipo, esc, pend, "primero", directo)
    if hay3 and s["puede_tercero"] and not s["ya_directo"]:
        apartado_terceros(equipo, esc, pend)

def panorama(equipos, jugados, esc, directo=None):
    directo = DIRECTO if directo is None else directo; hay3 = MEJORES_TERCEROS > 0
    filas = []
    for e in equipos:
        s = situacion(e, esc, directo)
        if s["ya_directo"]: est = "🟢 Clasificado directo"
        elif s["eliminado"]: est = "🔴 Eliminado"
        elif s["puede_directo"]: est = "🟡 En disputa"
        elif hay3: est = "🔵 Chance vía mejor 3º (depende de otros grupos)"
        else: est = "🔴 Eliminado"
        filas.append({"Equipo": e, "Estado": est, "Mejor": s["mejor"], "Peor": s["peor"],
                      "Puede 1º": "sí" if s["puede_1"] else "no", "Directo en": f"{s['ndir']}/{s['total']}"})
    orden = {r["Equipo"]: r["Pos"] for _, r in tabla(equipos, jugados).iterrows()}
    return pd.DataFrame(filas).sort_values("Equipo", key=lambda c: c.map(orden)).reset_index(drop=True)

# ---- puesto puntual ----
def _desc_obj(o):
    return {"exacto": f"exactamente {o[1]}º", "al_menos": f"{o[1]}º o mejor",
            "como_mucho": f"{o[1]}º o peor", "entre": f"entre {o[1]}º y {o[-1]}º"}[o[0]]
def _ok_pos(pos, o):
    if o[0] == "exacto": return pos == o[1]
    if o[0] == "al_menos": return pos <= o[1]
    if o[0] == "como_mucho": return pos >= o[1]
    return (pos >= o[1]) & (pos <= o[2])

def resultados_para_puesto(equipo, esc, pend, objetivo):
    pos = esc[f"Pos {equipo}"]; ok = _ok_pos(pos, objetivo); desc = _desc_obj(objetivo)
    n, tot = int(ok.sum()), len(esc)
    print(f"¿Qué resultados necesita {equipo} para terminar {desc}?\n")
    if n == 0:
        alc = ", ".join(f"{int(p)}º" for p in sorted(pos.unique()))
        print(f"❌ IMPOSIBLE: {equipo} no puede terminar {desc}."); print(f"   Puestos alcanzables: {alc}."); return
    if n == tot: print(f"✅ {equipo} termina {desc} pase lo que pase."); return
    df = esc.copy(); df["_c"] = df.apply(lambda r: _combo(r, pend), axis=1); df["_ok"] = ok.values
    siempre, aveces = [], []
    for c, g in df.groupby("_c"):
        k, m = int(g["_ok"].sum()), len(g)
        if k == m: siempre.append(c)
        elif k > 0: aveces.append((c, k, m))
    if siempre:
        print("Lo logra SIEMPRE con:")
        for c in siempre: print(f"  ✅ {c}")
    if aveces:
        print("\nLo logra SOLO si la dif. de gol acompaña:")
        for c, k, m in sorted(aveces, key=lambda x: -x[1] / x[2]): print(f"  ⚠️  {c}   ({k}/{m} marcadores)")

# ---- probabilidades (Monte Carlo Poisson) ----
def probabilidades(equipos, jugados, pendientes, n=8000, media=1.3, fuerza=None, seed=1):
    rng = np.random.default_rng(seed)
    lam = {e: media * (fuerza.get(e, 1.0) if fuerza else 1.0) for e in equipos}
    cuenta = {e: np.zeros(len(equipos) + 1, dtype=int) for e in equipos}; base = list(jugados)
    for _ in range(n):
        part = base + [(l, v, int(rng.poisson(lam[l])), int(rng.poisson(lam[v]))) for (l, v) in pendientes]
        for e, p in posiciones(equipos, part).items(): cuenta[e][p] += 1
    rows = [{"Equipo": e, "1º %": round(100 * cuenta[e][1] / n, 1),
             "Top 2 %": round(100 * cuenta[e][1:3].sum() / n, 1),
             "Top 3 %": round(100 * cuenta[e][1:4].sum() / n, 1)} for e in equipos]
    return pd.DataFrame(rows).sort_values("Top 2 %", ascending=False).reset_index(drop=True)

# ---- qué pasa si ----
def que_pasa_si(esc, pend, condiciones, equipos):
    mask = pd.Series(True, index=esc.index)
    for i, cond in enumerate(condiciones, 1):
        if not cond: continue
        gl, gv = esc[f"P{i}_gl"], esc[f"P{i}_gv"]
        mask &= (gl > gv) if cond == "L" else (gl == gv) if cond == "E" else (gl < gv)
    sub = esc[mask]
    rows = [{"Equipo": e, "Mejor": int(sub[f"Pos {e}"].min()), "Peor": int(sub[f"Pos {e}"].max()),
             "Directo posible": "sí" if (sub[f"Pos {e}"] <= 2).any() else "no",
             "Directo seguro": "sí" if (sub[f"Pos {e}"] <= 2).all() else "no"} for e in equipos]
    return sub, pd.DataFrame(rows)

# ---- distribución y casos extremos ----
def distribucion(equipos, esc):
    d = pd.DataFrame({e: esc[f"Pos {e}"].value_counts() for e in equipos}).fillna(0).astype(int).sort_index()
    d.index.name = "Puesto"; return d

def caso_extremo(equipo, esc, pend, mejor=True):
    pos = esc[f"Pos {equipo}"]; idx = pos.idxmin() if mejor else pos.idxmax(); row = esc.loc[idx]
    res = [(int(row[f"P{i}_gl"]), int(row[f"P{i}_gv"])) for i in range(1, len(pend) + 1)]
    print(("MEJOR" if mejor else "PEOR") + f" caso para {equipo}: termina {int(pos.loc[idx])}º")
    print("  " + texto_resultados(pend, res))
    return simular(ESTADO["equipos"], ESTADO["jugados"], pend, res)

print("Análisis cargado ✓")

In [ ]:
def mejor_resultado(equipo, esc, pend, directo=2):
    """Ordena los resultados propios de mejor a peor. Ganar nunca queda por debajo de empatar
       (ordena por posición promedio: más puntos nunca empeora la posición)."""
    df = esc.copy(); df["_p"] = df.apply(lambda r: _res_propio(r, equipo, pend), axis=1)
    rk = lambda p: 0 if p.startswith("le gana") else (1 if p.startswith("empata") else 2)
    opciones = []
    for prop, g in df.groupby("_p"):
        gp = esc.loc[g.index, f"Pos {equipo}"]
        opciones.append({"r": prop, "peor": int(gp.max()), "mejor": int(gp.min()),
                         "prom": float(gp.mean()), "uno": int((gp == 1).sum()),
                         "dir": int((gp <= directo).sum()), "n": len(g), "rk": rk(prop)})
    opciones.sort(key=lambda o: (round(o["prom"], 6), o["peor"], o["mejor"], o["rk"]))
    print(f"¿Qué le conviene a {equipo}? (de mejor a peor)\n")
    for i, o in enumerate(opciones):
        flag = "  👍 lo que más le conviene" if i == 0 else ""
        print(f"• Si {equipo} {o['r']}: termina entre {o['mejor']}º y {o['peor']}º · "
              f"sale 1º en {o['uno']}/{o['n']} · clasifica directo en {o['dir']}/{o['n']}{flag}")
    return opciones

def _gana_todo(p): return bool(p) and all(s.startswith("le gana") for s in p.split(" y "))

def conviene_otros(equipo, esc, pend, directo=None):
    """Qué le conviene al equipo en los partidos que NO juega (para qué hinchar)."""
    directo = DIRECTO if directo is None else directo
    if not [p for p in pend if equipo not in p]:
        return  # no hay otros partidos
    df = esc.copy()
    df["_p"] = df.apply(lambda r: _res_propio(r, equipo, pend), axis=1)
    df["_o"] = df.apply(lambda r: _res_otros(r, equipo, pend), axis=1)
    if _pd_de(equipo, pend):
        sub = df[df["_p"].map(_gana_todo)]
        cab = f"Si {equipo} gana lo suyo, le conviene en los OTROS partidos (de mejor a peor):"
        if sub.empty: sub, cab = df, f"A {equipo} le conviene en los OTROS partidos (de mejor a peor):"
    else:
        sub, cab = df, f"A {equipo} le conviene en los OTROS partidos (de mejor a peor):"
    rows = []
    for o, g in sub.groupby("_o"):
        gp = esc.loc[g.index, f"Pos {equipo}"]
        rows.append({"o": o, "prom": float(gp.mean()), "uno": int((gp == 1).sum()),
                     "dir": int((gp <= directo).sum()), "n": len(g)})
    rows.sort(key=lambda r: (round(r["prom"], 6), -r["dir"] / r["n"]))
    print(cab + "\n")
    for i, r in enumerate(rows):
        flag = "  👍" if i == 0 else ""
        print(f"• Que {r['o']}: sale 1º en {r['uno']}/{r['n']} · clasifica directo en {r['dir']}/{r['n']}{flag}")

def resumen_grupo(equipos, jugados, esc=None, pend=None, directo=None):
    """Un pantallazo en texto del grupo: líder, qué falta y cómo está la pelea."""
    directo = DIRECTO if directo is None else directo
    t = tabla(equipos, jugados); top = t.iloc[0]
    txt = f"📋 {top['Equipo']} lidera con {int(top['PTS'])} pts"
    if len(t) > 1: txt += f", escolta {t.iloc[1]['Equipo']} ({int(t.iloc[1]['PTS'])})."
    else: txt += "."
    partes = [txt]
    if pend: partes.append("Falta(n) " + ", ".join(f"{l}-{v}" for l, v in pend) + ".")
    if esc is not None:
        S = {e: situacion(e, esc, directo) for e in equipos}
        clas = [e for e in equipos if S[e]["ya_directo"]]
        elim = [e for e in equipos if S[e]["eliminado"]]
        disp = [e for e in equipos if not S[e]["ya_directo"] and not S[e]["eliminado"]]
        if clas: partes.append("Ya clasificó: " + ", ".join(clas) + ".")
        if elim: partes.append("Sin chances: " + ", ".join(elim) + ".")
        pelean = [e for e in disp if S[e]["puede_directo"]]
        if len(pelean) >= 2 and len(clas) < directo:
            partes.append(f"Pelean por entrar: {', '.join(pelean)}.")
        elif disp:
            partes.append("En disputa: " + ", ".join(disp) + ".")
    return " ".join(partes)

def _restantes(equipos, pend):
    r = {e: 0 for e in equipos}
    for l, v in pend: r[l] += 1; r[v] += 1
    return r

def maximos_minimos(equipos, jugados, pend):
    """Puntos actuales y máximos posibles. Sirve con cualquier cantidad de fechas restantes."""
    ov = _stats(equipos, jugados); rest = _restantes(equipos, pend)
    rows = [{"Equipo": e, "PJ": ov[e]["pj"], "PTS": ov[e]["pts"], "Restan": rest[e],
             "PTS máx": ov[e]["pts"] + 3 * rest[e]} for e in equipos]
    return pd.DataFrame(rows).sort_values(["PTS", "PTS máx"], ascending=False).reset_index(drop=True)

def clasificado_eliminado(equipos, jugados, pend, n=1):
    """Marca quién ya aseguró el top n y quién quedó sin chances (cuenta por puntos)."""
    ov = _stats(equipos, jugados); rest = _restantes(equipos, pend)
    pts = {e: ov[e]["pts"] for e in equipos}; pmax = {e: pts[e] + 3 * rest[e] for e in equipos}
    col = CAMPEON.capitalize() if n == 1 else f"Top {n}"
    rows = []
    for e in equipos:
        arriba = sum(1 for x in equipos if x != e and pmax[x] > pts[e])   # rivales que aún pueden superarlo
        inalc = sum(1 for x in equipos if x != e and pts[x] > pmax[e])    # rivales ya inalcanzables
        estado = "🟢 asegurado" if arriba < n else ("🔴 sin chances" if inalc >= n else "🟡 depende")
        rows.append({"Equipo": e, "PTS": pts[e], "PTS máx": pmax[e], col: estado})
    return pd.DataFrame(rows).sort_values("PTS", ascending=False).reset_index(drop=True)

def numero_magico(equipo, equipos, jugados, pend, n=1):
    """Puntos que faltan para ASEGURAR el top n (en el peor caso de los rivales). n=1 = campeón."""
    ov = _stats(equipos, jugados); rest = _restantes(equipos, pend)
    pts = {e: ov[e]["pts"] for e in equipos}; pmax = {e: pts[e] + 3 * rest[e] for e in equipos}
    otros = sorted((pmax[x] for x in equipos if x != equipo), reverse=True)
    meta = f"ser {CAMPEON}" if n == 1 else f"entrar al top {n}"
    print(f"{equipo} — para {meta} (asegurar, en el peor caso de los rivales):")
    print(f"  Tiene {pts[equipo]} pts y le quedan {rest[equipo]} partidos ({3*rest[equipo]} en juego).")
    if len(otros) < n:
        print(f"  ✅ Ya está en el top {n}."); return
    necesita = max(0, (otros[n-1] + 1) - pts[equipo]); tope = 3 * rest[equipo]
    if necesita == 0:
        print("  ✅ Ya está asegurado pase lo que pase.")
    elif necesita <= tope:
        print(f"  Necesita sumar {necesita} pts más para asegurarlo sin depender de nadie.")
    else:
        print(f"  No puede asegurarlo solo: necesitaría {necesita} y solo hay {tope} en juego → depende de que los rivales pinchen.")

def necesita_por_resultados(equipo, equipos, jugados, pendientes, n=None):
    """Para MUCHOS partidos: razona por resultados (G/E/P) y puntos, no por goles.
       Escenarios = 3^(partidos), mucho más livianos que simular marcadores."""
    n = DIRECTO if n is None else n
    if not pendientes:
        print("No quedan partidos."); return
    base = {e: _stats(equipos, jugados)[e]["pts"] for e in equipos}
    mios = [i for i, p in enumerate(pendientes) if equipo in p]
    otros = [i for i in range(len(pendientes)) if i not in mios]
    meta = f"ser {CAMPEON}" if n == 1 else f"clasificar (top {n})"
    verbo_ok = f"es {CAMPEON}" if n == 1 else f"entra al top {n}"
    porpts = {}
    for own in product("LEV", repeat=len(mios)):
        add = {e: 0 for e in equipos}
        for k, i in enumerate(mios):
            l, v = pendientes[i]
            if own[k] == "L": add[l] += 3
            elif own[k] == "V": add[v] += 3
            else: add[l] += 1; add[v] += 1
        pteam = base[equipo] + add[equipo]
        est = porpts.setdefault(pteam, [])
        for oth in product("LEV", repeat=len(otros)):
            final = {e: base[e] + add[e] for e in equipos}
            for k, i in enumerate(otros):
                l, v = pendientes[i]
                if oth[k] == "L": final[l] += 3
                elif oth[k] == "V": final[v] += 3
                else: final[l] += 1; final[v] += 1
            p = final[equipo]
            arriba = sum(1 for x in equipos if x != equipo and final[x] > p)
            igual = sum(1 for x in equipos if x != equipo and final[x] == p)
            rem = n - arriba
            est.append("safe" if rem >= igual + 1 else ("out" if rem <= 0 else "tie"))
    niveles = sorted(porpts, reverse=True)
    safe_pts = [p for p in niveles if all(s == "safe" for s in porpts[p])]
    out_pts = [p for p in niveles if all(s == "out" for s in porpts[p])]
    medio = [p for p in niveles if p not in safe_pts and p not in out_pts]
    print(f"¿Qué necesita {equipo} para {meta}? — por resultados ({3**len(pendientes)} combinaciones)\n")
    if safe_pts:
        print(f"✅ Con {min(safe_pts)} pts o más: {equipo} {verbo_ok} pase lo que pase.")
    if medio:
        borde = any("tie" in porpts[p] for p in medio)
        rng = f"{min(medio)} a {max(medio)}" if min(medio) != max(medio) else f"{medio[0]}"
        print(f"⚠️ Con {rng} pts: depende de los otros resultados" +
              (" (y en algunos casos de la diferencia de gol)" if borde else "") + ".")
    if out_pts:
        print(f"❌ Con {max(out_pts)} pts o menos: no le alcanza.")
    print("\n(Se razona por resultados; los empates de puntos por el último cupo se deciden por dif. de gol.)")

print("Funciones extra (qué conviene + cuentas de liga) cargadas ✓")

In [ ]:
ESTADO = {}      # equipos, jugados, pendientes, esc
REFRESCAR = []   # funciones que refrescan cada sección al cargar un grupo

def cargar_estado(equipos, jugados, pendientes, salida=None):
    mg = elegir_max_goles(len(pendientes))
    esc = todos_los_escenarios(equipos, jugados, pendientes, mg)
    ESTADO.update(equipos=equipos, jugados=jugados, pendientes=pendientes, esc=esc, mg=mg)
    if salida is not None:
        with salida:
            clear_output()
            print(f"{len(esc)} escenarios (máx {mg} goles/equipo).")
            if pendientes: print("Pendientes:", ", ".join(f"{l} vs {v}" for l, v in pendientes))
            print("\n" + resumen_grupo(equipos, jugados, esc, pendientes))
            print("\nTABLA ACTUAL"); display(tabla(equipos, jugados))
            print("PANORAMA"); display(panorama(equipos, jugados, esc))
    for f in REFRESCAR:
        try: f()
        except Exception as e: print("aviso:", e)

print("Estado compartido listo ✓")

In [ ]:
_MESES = r"(ene|feb|mar|abr|may|jun|jul|ago|sep|set|oct|nov|dic|jan|apr|aug|dec)"
_DIAS  = r"(lun|mar|mié|mie|jue|vie|sáb|sab|dom|mon|tue|wed|thu|fri|sat|sun)"
_RE_SCORE = re.compile(r"^(.+?)\s+(\d{1,2})\s*(?:[-–—xX]\s*(\d{1,2})|:\s*(\d))\s+(.+?)$")
_RE_VS    = re.compile(r"^(.+?)\s+(?:vs?\.?|–|—|-|x)\s+(.+?)$", re.I)

def _limpiar(ln):
    ln = ln.strip()
    pref = [rf"^{_DIAS}\w*\.?,?\s+", r"^\d{1,2}[:.]\d{2}\s+", r"^\d{1,2}[/\-.]\d{1,2}([/\-.]\d{2,4})?\s+",
            rf"^\d{{1,2}}\s+{_MESES}\w*\.?,?\s+", rf"^{_MESES}\w*\.?\s+\d{{1,2}},?\s+"]
    ch = True
    while ch:
        ch = False
        for p in pref:
            nu = re.sub(p, "", ln, flags=re.I)
            if nu != ln: ln = nu; ch = True
    ln = re.sub(r"\s*\(.*?\)\s*$", "", ln)
    ln = re.sub(r"\s*(FT|Finalizado|Final|Termin\w*|Ver resumen|Resumen)\s*$", "", ln, flags=re.I)
    return ln.strip()

def _norm(t): return re.sub(r"\s+", " ", t).strip(" -–—\t")
def _let(t): return bool(re.search(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]", t))

def parsear_resultados(texto):
    jug, pen, eq = [], [], []
    def add(t):
        if t and t not in eq: eq.append(t)
    for raw in texto.splitlines():
        ln = _limpiar(raw)
        if not ln: continue
        m = _RE_SCORE.match(ln)
        if m:
            loc, vis = _norm(m.group(1)), _norm(m.group(5))
            gl = int(m.group(2)); gv = int(m.group(3) if m.group(3) is not None else m.group(4))
            if _let(loc) and _let(vis): add(loc); add(vis); jug.append((loc, vis, gl, gv)); continue
        m = _RE_VS.match(ln)
        if m:
            loc, vis = _norm(m.group(1)), _norm(m.group(2))
            if _let(loc) and _let(vis) and not re.search(r"\d", loc + vis): add(loc); add(vis); pen.append((loc, vis))
    jp = {frozenset((l, v)) for l, v, _, _ in jug}; pp = {frozenset(p) for p in pen}
    for a, b in combinations(eq, 2):
        fs = frozenset((a, b))
        if fs not in jp and fs not in pp: pen.append((a, b)); pp.add(fs)
    return eq, jug, pen

_RE_HEADER = re.compile(r"^\s*(grupo|group|gpo)\s*[:.]?\s*([A-Za-z0-9]+)\s*$", re.I)
def dividir_grupos(texto):
    g, act, suelto = {}, None, []
    for ln in texto.splitlines():
        m = _RE_HEADER.match(ln.strip())
        if m: act = m.group(2).upper(); g.setdefault(act, [])
        else: (g[act] if act is not None else suelto).append(ln)
    if not g and any(s.strip() for s in suelto): g["Único"] = suelto
    return {k: "\n".join(v) for k, v in g.items()}

def analizar_torneo(texto, directo=None):
    directo = DIRECTO if directo is None else directo
    tablas, terceros, directos, avisos = {}, [], [], []
    for lab, txt in dividir_grupos(texto).items():
        eq, jug, pen = parsear_resultados(txt)
        if len(eq) < 3: avisos.append(f"Grupo {lab}: pocos equipos."); continue
        t = tabla(eq, jug); tablas[lab] = t
        if pen: avisos.append(f"Grupo {lab}: faltan {len(pen)} partido(s) → terceros provisorios.")
        for _, r in t.iterrows():
            if r["Pos"] <= directo: directos.append((lab, r["Equipo"], int(r["Pos"])))
            if r["Pos"] == 3: terceros.append((f"{lab} · {r['Equipo']}", int(r["PTS"]), int(r["DG"]), int(r["GF"])))
    def clave(t): return (t[1], t[2], t[3])
    tbl3 = (pd.DataFrame([{"Pos": i, "Grupo": t[0], "PTS": t[1], "DG": t[2], "GF": t[3],
                           "Clasifica": "✅ sí" if i <= MEJORES_TERCEROS else "❌ no"}
                          for i, t in enumerate(sorted(terceros, key=clave, reverse=True), 1)])
            if terceros and MEJORES_TERCEROS > 0 else None)
    return tablas, directos, tbl3, avisos

print("Parser + torneo cargados ✓")

In [ ]:
def _grp(lbl): return re.split(r"[ _]", str(lbl).strip())[-1].upper() if lbl else "?"
def _nom(t): return (t.get("shortName") or t.get("name") or t.get("tla") or "¿?").strip()
_FIN = {"FINISHED", "AWARDED"}

def matches_a_texto(matches):
    grupos = {}
    for m in matches:
        if "GROUP" not in str(m.get("stage", "")).upper() and not m.get("group"): continue
        g = _grp(m.get("group")); loc, vis = _nom(m["homeTeam"]), _nom(m["awayTeam"])
        ft = (m.get("score") or {}).get("fullTime") or {}; gl, gv = ft.get("home"), ft.get("away")
        if m.get("status") in _FIN and gl is not None and gv is not None:
            grupos.setdefault(g, []).append(f"{loc} {gl}-{gv} {vis}")
        else: grupos.setdefault(g, []).append(f"{loc} vs {vis}")
    out = []
    for g in sorted(grupos): out.append(f"Grupo {g}"); out.extend(grupos[g]); out.append("")
    return "\n".join(out).strip()

def traer_de_api(token, comp="WC"):
    import requests
    r = requests.get(f"https://api.football-data.org/v4/competitions/{comp}/matches",
                     headers={"X-Auth-Token": token}, timeout=30); r.raise_for_status()
    return r.json().get("matches", [])

def listar_competiciones(token):
    import requests
    r = requests.get("https://api.football-data.org/v4/competitions",
                     headers={"X-Auth-Token": token}, timeout=30); r.raise_for_status()
    return [(c.get("code"), c.get("name")) for c in r.json().get("competitions", [])]

print("API lista ✓")

## 1) 🌐 Cargar todo automático (API)

Registrate gratis en `https://www.football-data.org/client/register`, copiá tu **API key**,
pegala abajo, dejá `WC` (Mundial) y apretá **Traer datos**. Elegí después un grupo para
analizarlo en las secciones siguientes. (Si `WC` no trae nada, usá **ver torneos**.)

In [ ]:
def construir_api():
    token = W.Text(description="API key:", placeholder="football-data.org", layout=W.Layout(width="430px"))
    comp = W.Text(value="WC", description="Torneo:", layout=W.Layout(width="200px"))
    fetch = W.Button(description="🌐 Traer datos", button_style="warning")
    vert = W.Button(description="ver torneos", layout=W.Layout(width="120px"))
    grupo = W.Dropdown(description="Analizar grupo:", layout=W.Layout(width="300px"))
    out = W.Output(); cache = {"texto": ""}

    def cargar(texto):
        cache["texto"] = texto
        if not texto.strip():
            with out: clear_output(); print("No vinieron partidos de fase de grupos.")
            return
        if "_set_torneo" in globals(): globals()["_set_torneo"](texto)
        grupo.options = list(dividir_grupos(texto).keys())
        with out:
            clear_output(); print("Datos cargados ✓  Elegí un grupo arriba.")
            print("Grupos:", ", ".join(grupo.options))
        if grupo.options: grupo.value = grupo.options[0]

    def on_fetch(_):
        with out:
            clear_output()
            if not token.value.strip(): print("Pegá tu API key."); return
            print("Trayendo…")
        try: matches = traer_de_api(token.value.strip(), comp.value.strip() or "WC")
        except Exception as e:
            with out: clear_output(); print("No pude traer los datos:", e); print("Probá 'ver torneos'.")
            return
        cargar(matches_a_texto(matches))

    def on_ver(_):
        with out:
            clear_output()
            try:
                for c, n in listar_competiciones(token.value.strip()): print(f"  {c} — {n}")
            except Exception as e: print("Error:", e)

    def on_grupo(_):
        if not grupo.value or not cache["texto"]: return
        eq, jug, pen = parsear_resultados(dividir_grupos(cache["texto"]).get(grupo.value, ""))
        if len(eq) >= 3: cargar_estado(eq, jug, pen, out)

    fetch.on_click(on_fetch); vert.on_click(on_ver); grupo.observe(on_grupo, "value")
    display(W.HBox([token, vert]), W.HBox([comp, fetch]), grupo, out)

if _HAY_WIDGETS: construir_api()

## 2) ❓ ¿Qué necesita un equipo? (para distintas situaciones)

Elegí la **situación**: ser campeón (1º), clasificar (top N), entrar a Champions (top 4),
evitar el descenso (últimos N) o el reporte completo. Te muestra, según gane/empate/pierda,
si lo logra **seguro**, **depende** o es **imposible**, y de qué depende. (Cargá un grupo en la sección 1.)

In [ ]:
def construir_necesita():
    eq = W.Dropdown(description="Equipo:")
    opts = [("Reporte completo","full"), (f"Ser {CAMPEON} (1º)","campeon"),
            ("Clasificar directo (top N)","top"), ("Entrar a Champions (top 4)","champions"),
            ("Evitar el descenso (últimos N)","descenso")]
    if MEJORES_TERCEROS > 0: opts.append(("Quedar 3º (mejor tercero)","tercero"))
    obj = W.Dropdown(description="Situación:", style={"description_width":"initial"},
        layout=W.Layout(width="320px"), options=opts)
    n = W.BoundedIntText(value=DIRECTO, min=1, max=20, description="N:", layout=W.Layout(width="130px"))
    out = W.Output()
    def render(*_):
        n.layout.display = "" if obj.value in ("top","descenso") else "none"
        with out:
            clear_output()
            if "esc" not in ESTADO: print("Cargá un grupo en la sección 1."); return
            if not eq.value: return
            E, P = ESTADO["esc"], ESTADO["pendientes"]
            if obj.value == "full": analizar(eq.value, E, P)
            elif obj.value == "champions": que_necesita(eq.value, E, P, "top", n=4)
            elif obj.value == "tercero": apartado_terceros(eq.value, E, P)
            elif obj.value in ("top","descenso"): que_necesita(eq.value, E, P, obj.value, n=n.value)
            else: que_necesita(eq.value, E, P, obj.value)
            print("TABLA ACTUAL (para ubicarse)")
            display(tabla(ESTADO["equipos"], ESTADO["jugados"]))
    def refrescar():
        eq.options = ESTADO.get("equipos", [])
        if eq.options: eq.value = eq.options[0]
        render()
    REFRESCAR.append(refrescar)
    for w in (eq, obj, n): w.observe(render, "value")
    display(W.HBox([eq, obj, n]), out)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_necesita()

## 3) 🎯 Elegir un puesto y ver qué resultados se necesitan

Elegí un equipo y un **puesto**. Te muestra los resultados con los que lo consigue (seguro,
o si depende de la diferencia de gol) y **te avisa si es imposible**, diciéndote a qué
puestos sí puede llegar.

In [ ]:
def construir_puesto():
    eq = W.Dropdown(description="Equipo:")
    puesto = W.Dropdown(description="Puesto:")
    modo = W.RadioButtons(options=[("Exactamente ese puesto", "exacto"),
                                   ("Ese puesto o mejor", "al_menos")], description="Modo:")
    out = W.Output()
    def render(*_):
        with out:
            clear_output()
            if "esc" not in ESTADO: print("Cargá un grupo en la sección 1."); return
            if not eq.value or not puesto.value: return
            resultados_para_puesto(eq.value, ESTADO["esc"], ESTADO["pendientes"], (modo.value, int(puesto.value)))
    def refrescar():
        eqs = ESTADO.get("equipos", [])
        eq.options = eqs; puesto.options = [(f"{i}º", i) for i in range(1, len(eqs) + 1)]
        if eqs: eq.value = eqs[0]; puesto.value = 2
        render()
    REFRESCAR.append(refrescar)
    for w in (eq, puesto, modo): w.observe(render, "value")
    display(W.HBox([eq, puesto, modo]), out)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_puesto()

## 4) 🔀 ¿Qué pasa si…?

Fijá el resultado de uno o más partidos pendientes y mirá cómo queda el grupo.

In [ ]:
def construir_quepasasi():
    cont = W.VBox(); out = W.Output(); btn = W.Button(description="Ver", button_style="primary")
    refs = {"items": []}
    def refrescar():
        refs["items"] = []; rows = []
        for (l, v) in ESTADO.get("pendientes", []):
            dd = W.Dropdown(options=[("(cualquiera)", None), (f"gana {l}", "L"),
                ("empate", "E"), (f"gana {v}", "V")], description=f"{l} vs {v}:",
                style={"description_width": "initial"}, layout=W.Layout(width="320px"))
            rows.append(dd); refs["items"].append(dd)
        cont.children = rows if rows else [W.Label("Cargá un grupo en la sección 1.")]
        with out: clear_output()
    def ver(_):
        with out:
            clear_output()
            if "esc" not in ESTADO: print("Cargá un grupo en la sección 1."); return
            cond = [dd.value for dd in refs["items"]]
            sub, resumen = que_pasa_si(ESTADO["esc"], ESTADO["pendientes"], cond, ESTADO["equipos"])
            print(f"{len(sub)} escenarios cumplen esa condición."); display(resumen)
    btn.on_click(ver); REFRESCAR.append(refrescar)
    display(cont, btn, out)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_quepasasi()

## 5) 🎲 Probabilidades estimadas

Estima la chance de cada equipo simulando los partidos pendientes con goles al azar (modelo
de Poisson). **Es un modelo simple y neutral** (no pondera el nivel de cada rival), así que
tomalo como orientación; lo exacto son las secciones 2–4.

In [ ]:
def construir_prob():
    media = W.FloatSlider(value=1.3, min=0.5, max=3.0, step=0.1, description="Goles prom.:",
                          style={"description_width": "initial"}, readout_format=".1f")
    n = W.IntText(value=8000, description="Simulaciones:", style={"description_width": "initial"})
    btn = W.Button(description="Estimar", button_style="success"); out = W.Output()
    def run(_):
        with out:
            clear_output()
            if "esc" not in ESTADO: print("Cargá un grupo en la sección 1."); return
            print("Simulando…")
            df = probabilidades(ESTADO["equipos"], ESTADO["jugados"], ESTADO["pendientes"],
                                n=int(n.value), media=media.value)
            clear_output(); display(df)
    btn.on_click(run); REFRESCAR.append(lambda: out.clear_output())
    display(W.HBox([media, n, btn]), out)

if _HAY_WIDGETS: construir_prob()

## 6) 📈 Distribución de puestos y mejor/peor caso

In [ ]:
def construir_dist():
    out1 = W.Output(); eq = W.Dropdown(description="Equipo:")
    cual = W.RadioButtons(options=[("Mejor caso", True), ("Peor caso", False)], description="Ver:")
    out2 = W.Output()
    def refrescar():
        eq.options = ESTADO.get("equipos", [])
        with out1:
            clear_output()
            if "esc" in ESTADO:
                print("En cuántos escenarios cae cada equipo en cada puesto:")
                display(distribucion(ESTADO["equipos"], ESTADO["esc"]))
        if eq.options: eq.value = eq.options[0]
        ver()
    def ver(*_):
        with out2:
            clear_output()
            if "esc" not in ESTADO or not eq.value: return
            display(caso_extremo(eq.value, ESTADO["esc"], ESTADO["pendientes"], mejor=cual.value))
    eq.observe(ver, "value"); cual.observe(ver, "value"); REFRESCAR.append(refrescar)
    display(out1, W.HBox([eq, cual]), out2)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_dist()

## 7) 🏆 Torneo completo y mejores terceros

Si cargaste por API, ya tenés acá el torneo entero. También podés pegar los grupos a mano
(encabezá cada uno con `Grupo X`).

In [ ]:
def construir_torneo():
    box = W.Textarea(description="Grupos:", layout=W.Layout(width="480px", height="220px"),
                     placeholder="Grupo A\nEquipo1 1-0 Equipo2\n...")
    btn = W.Button(description="Analizar torneo", button_style="success"); out = W.Output()
    def run(_):
        with out:
            clear_output()
            tablas, directos, tbl3, avisos = analizar_torneo(box.value)
            if not tablas: print("Pegá los grupos (encabezados 'Grupo X') o cargá por API."); return
            for lab, t in tablas.items(): print(f"— Grupo {lab} —"); display(t)
            print("DIRECTOS:", " · ".join(f"{g}{p}º {e}" for g, e, p in directos))
            if tbl3 is not None: print("\nMEJORES TERCEROS (✅ entra entre los 8):"); display(tbl3)
            if avisos: print("Notas:"); [print("  -", a) for a in avisos]
    btn.on_click(run)
    globals()["_set_torneo"] = lambda t: (setattr(box, "value", t), run(None))
    display(box, btn, out)

if _HAY_WIDGETS: construir_torneo()

## 8) 🧮 Simular un resultado puntual

In [ ]:
def construir_sim():
    cont = W.VBox(); out = W.Output(); btn = W.Button(description="Simular", button_style="primary")
    refs = {"items": []}
    def refrescar():
        refs["items"] = []; rows = []
        for (l, v) in ESTADO.get("pendientes", []):
            wl = W.BoundedIntText(value=0, min=0, max=20, layout=W.Layout(width="55px"))
            wv = W.BoundedIntText(value=0, min=0, max=20, layout=W.Layout(width="55px"))
            rows.append(W.HBox([W.Label(l, layout=W.Layout(width="130px")), wl, W.Label("-"), wv,
                                W.Label(v, layout=W.Layout(width="130px"))]))
            refs["items"].append((l, v, wl, wv))
        cont.children = rows if rows else [W.Label("Cargá un grupo en la sección 1.")]
    def sim(_):
        with out:
            clear_output()
            if "esc" not in ESTADO: print("Cargá un grupo en la sección 1."); return
            res = [(wl.value, wv.value) for _, _, wl, wv in refs["items"]]
            print(texto_resultados(ESTADO["pendientes"], res))
            display(simular(ESTADO["equipos"], ESTADO["jugados"], ESTADO["pendientes"], res))
    btn.on_click(sim); REFRESCAR.append(refrescar)
    display(cont, btn, out)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_sim()

## 9) ✍️ Cargar sin API (pegar o a mano)

Si no usás la API: pegá la **lista de resultados** de un grupo y apretá **Cargar**. Detecta
equipos, jugados y deduce los pendientes. Eso alimenta todas las secciones de arriba.

In [ ]:
def construir_pegar():
    box = W.Textarea(description="Pegar:", layout=W.Layout(width="430px", height="130px"),
                     placeholder="España 0-0 Cabo Verde\nUruguay 1-1 Arabia Saudita\n...")
    btn = W.Button(description="📋 Cargar", button_style="warning"); out = W.Output()
    def run(_):
        eq, jug, pen = parsear_resultados(box.value)
        if len(eq) < 3:
            with out: clear_output(); print("Pegá al menos un grupo. Ej: 'España 0-0 Cabo Verde'.")
            return
        cargar_estado(eq, jug, pen, out)
    btn.on_click(run); display(box, btn, out)

if _HAY_WIDGETS: construir_pegar()

## 10) 🔧 Desempate / adaptar a otros torneos

Cambiá el criterio de desempate y la calculadora recalcula todo. **Olímpico** (mano a mano
primero) sirve para Mundial, Eurocopa, La Liga, Serie A. **Diferencia de gol primero** sirve
para Premier League, Bundesliga y la fase liga de Champions. Para una **liga** completa, en la
sección 1 cargá todos los partidos (los que faltan quedan como pendientes) y mirá la sección 11.

Además podés adaptar la **estructura de clasificación** ejecutando en una celda:
`DIRECTO = 2` (cuántos pasan directo por grupo) y `MEJORES_TERCEROS = 8` (cuántos mejores
terceros entran en el torneo; poné **`MEJORES_TERCEROS = 0`** si los terceros NO clasifican,
p. ej. un grupo donde solo pasan los 2 primeros). El análisis muestra o esconde el apartado de
mejores terceros automáticamente según ese valor.

In [ ]:
def construir_desempate():
    dd = W.Dropdown(options=list(PRESETS.keys()), description="Desempate:",
                    style={"description_width":"initial"}, layout=W.Layout(width="560px"))
    out = W.Output()
    def cambiar(*_):
        usar_desempate(PRESETS[dd.value])
        with out:
            clear_output(); print("Desempate aplicado:", dd.value)
        if "esc" in ESTADO:
            cargar_estado(ESTADO["equipos"], ESTADO["jugados"], ESTADO["pendientes"])
    dd.observe(cambiar, "value")
    display(dd, out)

if _HAY_WIDGETS: construir_desempate()

## 11) 🟰 ¿Qué le conviene a cada equipo?

El resultado propio (ganar / empatar / perder) ordenado de mejor a peor para su clasificación.

In [ ]:
def construir_conviene():
    eq = W.Dropdown(description="Equipo:"); out = W.Output()
    def render(*_):
        with out:
            clear_output()
            if "esc" not in ESTADO: print("Cargá un grupo en la sección 1."); return
            if eq.value:
                mejor_resultado(eq.value, ESTADO["esc"], ESTADO["pendientes"])
                print()
                conviene_otros(eq.value, ESTADO["esc"], ESTADO["pendientes"])
    def refrescar():
        eq.options = ESTADO.get("equipos", [])
        if eq.options: eq.value = eq.options[0]
        render()
    REFRESCAR.append(refrescar); eq.observe(render, "value")
    display(eq, out)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_conviene()

## 12) 🏆 Campeón / Champions / cuentas de liga

Cuentas que se usan seguido y que **sirven con muchas fechas por jugar** (no simulan goles):
puntos máximos posibles, quién ya **aseguró** o quedó **sin chances** para el top N, y el
**número mágico** (puntos para asegurar). Poné `n=1` para **campeón**, `n=4` para **Champions**,
`n=2` para clasificación directa de grupo, etc.

> El 1º se llama **campeón** por defecto. Si estás analizando un **grupo o zona**, podés
> cambiar la palabra ejecutando `CAMPEON = "1º de la zona"` (o `"ganador del grupo"`) y se
> usará en todos los textos.

In [ ]:
def construir_liga():
    n = W.BoundedIntText(value=1, min=1, max=12, description="Top N (1=campeón):",
                         style={"description_width":"initial"}, layout=W.Layout(width="230px"))
    eq = W.Dropdown(description="Equipo:")
    btn = W.Button(description="Calcular", button_style="success"); out = W.Output()
    def run(*_):
        with out:
            clear_output()
            if "esc" not in ESTADO: print("Cargá un grupo/liga en la sección 1."); return
            E, J, P = ESTADO["equipos"], ESTADO["jugados"], ESTADO["pendientes"]
            print("PUNTOS MÁXIMOS POSIBLES"); display(maximos_minimos(E, J, P))
            print(f"ASEGURADO / SIN CHANCES (top {n.value})"); display(clasificado_eliminado(E, J, P, n.value))
            if eq.value: numero_magico(eq.value, E, J, P, n.value)
    def refrescar():
        eq.options = ESTADO.get("equipos", [])
        if eq.options: eq.value = eq.options[0]
        with out: clear_output()
    btn.on_click(run); n.observe(run,"value"); eq.observe(run,"value"); REFRESCAR.append(refrescar)
    display(W.HBox([n, eq, btn]), out)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_liga()

## 13) 🗓️ ¿Y si faltan muchos partidos? (por resultados)

Cuando quedan 3, 4, 5 o más partidos, simular gol por gol explota. Acá se razona por
**resultados** (ganar/empatar/perder) y **puntos**: te dice, según con cuántos puntos termine
el equipo, si **clasifica seguro**, **depende de los otros** o **no le alcanza**. Los empates de
puntos por el último cupo quedan marcados como *definidos por diferencia de gol*. Para una liga
con muchísimas fechas, sumá la sección 12 (número mágico, asegurados).

In [ ]:
def construir_resultados():
    eq = W.Dropdown(description="Equipo:")
    n = W.BoundedIntText(value=DIRECTO, min=1, max=12, description="Top N:",
                         style={"description_width":"initial"}, layout=W.Layout(width="160px"))
    btn = W.Button(description="Calcular", button_style="success"); out = W.Output()
    def run(*_):
        with out:
            clear_output()
            if "equipos" not in ESTADO: print("Cargá un grupo en la sección 1."); return
            if eq.value:
                necesita_por_resultados(eq.value, ESTADO["equipos"], ESTADO["jugados"], ESTADO["pendientes"], n.value)
    def refrescar():
        eq.options = ESTADO.get("equipos", [])
        if eq.options: eq.value = eq.options[0]
        with out: clear_output()
    btn.on_click(run); eq.observe(run, "value"); n.observe(run, "value"); REFRESCAR.append(refrescar)
    display(W.HBox([eq, n, btn]), out)
    if ESTADO.get("equipos"): refrescar()

if _HAY_WIDGETS: construir_resultados()

## (Opcional) Versión por código

In [ ]:
equipos = ["España", "Uruguay", "Cabo Verde", "Arabia Saudita"]
jugados = [("España","Cabo Verde",0,0),("España","Arabia Saudita",4,0),
           ("Uruguay","Cabo Verde",2,2),("Uruguay","Arabia Saudita",1,1)]
pendientes = [("Uruguay","España"),("Cabo Verde","Arabia Saudita")]
esc = todos_los_escenarios(equipos, jugados, pendientes, 5)
ESTADO.update(equipos=equipos, jugados=jugados, pendientes=pendientes, esc=esc)

display(tabla(equipos, jugados)); display(panorama(equipos, jugados, esc))
resultados_para_puesto("Cabo Verde", esc, pendientes, ("exacto", 2))
print(); print(probabilidades(equipos, jugados, pendientes, n=4000).to_string(index=False))
print(); mejor_resultado("Uruguay", esc, pendientes)
print(); que_necesita("Uruguay", esc, pendientes, "campeon")
print(); que_necesita("Cabo Verde", esc, pendientes, "top", n=2)
print(); display(maximos_minimos(equipos, jugados, pendientes))
print(); numero_magico("España", equipos, jugados, pendientes, n=1)

# --- Ejemplo OTRO TORNEO: mini-liga con desempate por diferencia de gol (estilo Premier/Champions) ---
print("\n--- Ejemplo liga (desempate por diferencia de gol) ---")
usar_desempate(PRESETS["Diferencia de gol primero (Premier, Bundesliga, Champions fase liga)"])
ligaE = ["A","B","C","D"]
ligaJ = [("A","B",2,0),("C","D",1,1),("A","C",1,0),("B","D",3,1)]
ligaP = [("A","D"),("B","C")]
display(tabla(ligaE, ligaJ))
numero_magico("A", ligaE, ligaJ, ligaP, n=1)
display(clasificado_eliminado(ligaE, ligaJ, ligaP, n=2))
usar_desempate(PRESETS["Olímpico — mano a mano primero (FIFA, Euro, La Liga, Serie A)"])

# --- MUCHOS PARTIDOS: por resultados (sin simular goles) ---
print("\n--- Muchos partidos: análisis por resultados ---")
gE = ["A", "B", "C", "D"]
gJ = [("A", "B", 1, 0), ("C", "D", 0, 0)]          # 1 jugado por equipo
gP = [("A", "C"), ("B", "D"), ("A", "D"), ("B", "C")]  # 4 pendientes (cada equipo juega 2)
necesita_por_resultados("A", gE, gJ, gP, n=2)

---
### Notas
- **Desempate olímpico (FIFA 2026):** puntos → mano a mano (pts/dg/goles) → dg general →
  goles → fair play → ranking. Si ni eso separa, sorteo (acá, orden fijo).
- **API:** el conversor está hecho para football-data.org (campo `group`, `status`,
  `score.fullTime`). Para otra API puede necesitar ajustes en `matches_a_texto`.
- **Probabilidades:** modelo de Poisson neutral; orientativo, no oficial.
- **Rendimiento:** con muchos pendientes baja solo el máximo de goles simulados.